## Preprocessing

In [ ]:
pip install miceforest

In [1]:
# Import neccessary libraries.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#from miceforest import miceforest as mf
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn import set_config
from sklearn.model_selection import train_test_split

from sksurv.ensemble import RandomSurvivalForest

set_config(display="text")  # displays text representation of estimators

In [2]:
df_train = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')

In [3]:
# identifying garbage values (form of object data type)
for i in df_train.select_dtypes(include='object').columns:
    print(df_train[i].value_counts()) # Give the count of each unique value in the column.
    print("***"*10)

dri_score
Intermediate                                         10436
N/A - pediatric                                       4779
High                                                  4701
N/A - non-malignant indication                        2427
TBD cytogenetics                                      2003
Low                                                   1926
High - TED AML case <missing cytogenetics             1414
Intermediate - TED AML case <missing cytogenetics      481
N/A - disease not classifiable                         272
Very high                                              198
Missing disease status                                   9
Name: count, dtype: int64
******************************
psych_disturb
No          23005
Yes          3587
Not done      146
Name: count, dtype: int64
******************************
cyto_score
Poor            8802
Intermediate    6376
Favorable       3011
TBD             1341
Normal           643
Other            504
Not tested        55
N

In [4]:
def le(df: pd.DataFrame) -> pd.DataFrame:
    """Label encode the 'Yes' and 'No' values in the dataset."""
    # Base case.
    df['melphalan_dose'] = df['melphalan_dose'].map({'N/A, Mel not given': 0, 'MEL': 1})
    df['mrd_hct'] = df['mrd_hct'].map({'Negative': 0, 'Positive': 1})

    # Init.
    yes_no_columns = ['psych_disturb', 'diabetes', 'arrhythmia', 'vent_hist', 'renal_issue', 'pulm_severe', 'rituximab', 'obesity', 'in_vivo_tcd', 'hepatic_severe', 'prior_tumor', 'peptic_ulcer', 'rheum_issue', 'hepatic_mild', 'cardiac', 'pulm_moderate']

    # Treatment.
    for col in yes_no_columns:
        # Map the values in integer form.
        df[col] = df[col].map({'Yes': 1, 'No': 0})
    
    return df

def not_tested(df: pd.DataFrame) -> pd.DataFrame:
    """Treat the "Not tested" values in the dataset as missing values."""
    # Treatment.
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].replace('Not tested', np.nan)
    
    return df

def mice(df: pd.DataFrame) -> pd.DataFrame:
    """Impute missing values using the MICE algorithm."""
    # Init.
    numerical_features = df.select_dtypes(include=[np.number])

    # Treatment.
    kernel = mf.ImputationKernel(
        numerical_features,
        num_datasets=10,
        random_state=42
    )

    kernel.mice(5) # 5 iterations of imputation.

    imputed_numerical_features = kernel.complete_data()

    # Update the dataframe.
    df[numerical_features.columns] = imputed_numerical_features[numerical_features.columns]

    return df

def cat_imputing(df: pd.DataFrame, cat_type: None) -> pd.DataFrame:
    """Impute missing mode/'unknown'/dropping for categorical values"""
    # Treatment.
    if cat_type == 'mode':
        df.fillna(df.mode().iloc[0], inplace=True)
    elif cat_type == 'median':
        df.fillna(df.median().iloc[0], inplace=True)
    else:
        df.fillna('unknown', inplace=True)

    
    return df

In [5]:
def preprocessing(df: pd.DataFrame, drop=None) -> pd.DataFrame:
    """Preprocess the dataset."""
    # Init.
    #if drop: df.dropna(axis=1, thresh=0.5*df.shape[0], inplace=True)

    # Treatment.
    df = le(df)
    df = not_tested(df)
    #df = mice(df)
    df = cat_imputing(df, cat_type="mode")
    
    return df

In [6]:
df_train = preprocessing(df_train, drop=True)
df_test = preprocessing(df_test, drop=True)

## Processing

Random Survival Forest

In [7]:
print(df_train.shape)
print(df_test.shape)
df_train.head()

(28800, 60)
(3, 58)


,ID,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,...,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs,efs_time
0,0,N/A - non-malignant indication,0.0,Poor,0.0,2.0,8.0,No TBI,0.0,6.0,...,Permissive mismatched,Unrelated,0.0,8.0,0.0,2.0,0.0,10.0,0.0,42.356
1,1,Intermediate,0.0,Intermediate,0.0,2.0,8.0,"TBI +- Other, >cGy",0.0,6.0,...,Permissive mismatched,Related,0.0,8.0,0.0,2.0,1.0,10.0,1.0,4.672
2,2,N/A - non-malignant indication,0.0,Poor,0.0,2.0,8.0,No TBI,0.0,6.0,...,Permissive mismatched,Related,0.0,8.0,0.0,2.0,0.0,10.0,0.0,19.793
3,3,High,0.0,Intermediate,0.0,2.0,8.0,No TBI,0.0,6.0,...,Permissive mismatched,Unrelated,0.0,8.0,0.0,2.0,0.0,10.0,0.0,102.349
4,4,High,0.0,Poor,0.0,2.0,8.0,No TBI,0.0,6.0,...,Permissive mismatched,Related,1.0,8.0,0.0,2.0,0.0,10.0,0.0,16.223


In [ ]:
# One-hot encode categorical columns in 0/1 format.
df_train = pd.get_dummies(df_train, dtype=int)
df_test = pd.get_dummies(df_test, dtype=int)

# Add missing columns in the test set that are present in the train set and set them to 0.
missing_cols = set(df_train.columns) - set(df_test.columns)
for col in missing_cols:
    df_test[col] = 0

print(df_train.shape)
print(df_test.shape)

In [19]:
X_train, X_test = df_train.drop(['ID', 'efs', 'efs_time'], axis=1), df_test.drop(['ID'], axis=1)
y = np.array([tuple(row) for row in df_train[['efs', 'efs_time']].values], dtype=[('cens', '?'), ('time', '<f8')])

In [20]:
print(df_train.shape)
print(df_test.shape)

(28800, 21202)
(3, 81)


In [ ]:
random_state = 20

rsf = RandomSurvivalForest(n_estimators=100,
                           min_samples_split=10,
                           min_samples_leaf=15,
                           n_jobs=-1,
                           random_state=random_state)
rsf.fit(X_train, y)

In [ ]:
pd.Series(rsf.predict(X_test))
#prediction = pd.Series(rsf.predict(X_test))